<a href="https://colab.research.google.com/github/aabyyaann/midterm-machine-learning/blob/main/UTS_ML_NaufalAlifAbyan_101032300032.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# UTS MACHINE LEARNING - FRAUD DETECTION
# Nama: Naufal Alif Abyan | NIM: 101032300032 | TK-46-GAB

# Instalasi library pendukung
!pip install mlflow optuna lightgbm -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import optuna
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import os

print("Library Berhasil Dimuat.")

In [32]:
def find_path(name):
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if name in files:
            return root + "/"
    return None

# Mencari lokasi folder dataset Naufal secara otomatis
DETECTED_PATH = find_path('train_transaction.csv')

if DETECTED_PATH:
    print(f"File ditemukan di: {DETECTED_PATH}")
    # Membaca Data
    df = pd.read_csv(DETECTED_PATH + 'train_transaction.csv')

    # Fungsi optimasi memori agar Colab tidak crash
    for col in df.columns:
        if df[col].dtype == 'float64': df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64': df[col] = df[col].astype('int32')

    print("Data Transaction Berhasil Dimuat.")
else:
    print("File TIDAK ditemukan. Pastikan file 'train_transaction.csv' ada di Drive kamu.")

File ditemukan di: /content/drive/MyDrive/Midterm ML/
Data Transaction Berhasil Dimuat.


In [ ]:
# 1. Preprocessing Sederhana (Mengambil 50 kolom pertama agar cepat)
X = df.iloc[:, :50].drop(['isFraud', 'TransactionID'], axis=1, errors='ignore')
y = df['isFraud']

# Label Encoding untuk kolom kategori
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = X[col].fillna('Unknown')
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    else:
        X[col] = X[col].fillna(X[col].median())

# Split Data (80% Train, 20% Validasi)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 2. Optimization dengan Optuna
def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 31, 64),
        'n_estimators': 50
    }

    with mlflow.start_run(nested=True):
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, preds)
        return auc

print("Memulai Pencarian Parameter Terbaik...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

print(f"Best Score: {study.best_value}")

[I 2026-05-14 11:43:33,447] A new study created in memory with name: no-name-8086800f-cf7b-48df-90b6-6e3a3635e164


Memulai Pencarian Parameter Terbaik...


2026/05/14 11:43:38 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/14 11:43:38 INFO mlflow.store.db.utils: Updating database tables
[I 2026-05-14 11:43:55,266] Trial 0 finished with value: 0.9075124123337714 and parameters: {'learning_rate': 0.06738634300251291, 'num_leaves': 41}. Best is trial 0 with value: 0.9075124123337714.
[I 2026-05-14 11:44:04,062] Trial 1 finished with value: 0.908006332756742 and parameters: {'learning_rate': 0.05331789905512821, 'num_leaves': 61}. Best is trial 1 with value: 0.908006332756742.
[I 2026-05-14 11:44:11,623] Trial 2 finished with value: 0.9065681690290494 and parameters: {'learning_rate': 0.07871595069691519, 'num_leaves': 35}. Best is trial 1 with value: 0.908006332756742.


In [ ]:
# Latih model final dengan parameter terbaik hasil Optuna
best_model = lgb.LGBMClassifier(**study.best_params)
best_model.fit(X_train, y_train)

# Prediksi
y_pred_prob = best_model.predict_proba(X_val)[:, 1]
y_pred_bin = (y_pred_prob > 0.5).astype(int)

# Visualisasi Confusion Matrix
plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_val, y_pred_bin), annot=True, fmt='d', cmap='Blues')
plt.title(f'Fraud Detection CM - {NIM}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\nClassification Report:")
print(classification_report(y_val, y_pred_bin))